# Helper

In [1]:
import torch
import os
import json
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torch.utils.data import DataLoader, random_split

def load_weights(model, weight_path, strict=False):
    weight_dict = torch.load(weight_path, map_location='cpu')

    if 'model_state_dict' in weight_dict:
        weight_dict = weight_dict['model_state_dict']

    first_key = next(iter(weight_dict))
    if first_key.startswith('backbone.'):
        missing, unexpected = model.load_state_dict(weight_dict, strict=strict)
    else:
        missing, unexpected = model.backbone.load_state_dict(weight_dict, strict=strict)

    if missing:
        print(f"Missing keys: {missing}")
    if unexpected:
        print(f"Unexpected keys: {unexpected}")

    return model

def get_dataloader(dataset_path, config):
    transform = Compose([
        Resize((224, 224)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225])
    ])

    dataset = ImageFolder(root=dataset_path, transform=transform)

    os.makedirs(config['save_dir'], exist_ok=True)
    mapping_path = os.path.join(config['save_dir'], f"{config['run_name']}_class_mapping.json")
    with open(mapping_path, 'w') as f:
        json.dump(dataset.class_to_idx, f, indent=2)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)

    return train_loader, val_loader, dataset.class_to_idx


# Model

In [2]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Optional, Callable, List, Type, Union

def conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        groups=groups,
        bias=False,
        dilation=dilation,
    )


def conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion: int = 1

    def __init__(
        self,
        inplanes: int,
        planes: int,
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
        groups: int = 1,
        base_width: int = 64,
        dilation: int = 1,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class Bottleneck(nn.Module):
    expansion: int = 4

    def __init__(
        self,
        inplanes: int,
        planes: int,
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
        groups: int = 1,
        base_width: int = 64,
        dilation: int = 1,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        width = int(planes * (base_width / 64.0)) * groups
        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        layers: list[int],
        num_classes: int = 1000,
        zero_init_residual: bool = False,
        groups: int = 1,
        width_per_group: int = 64,
        replace_stride_with_dilation: Optional[list[bool]] = None,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                f"or a 3-element tuple, got {replace_stride_with_dilation}"
            )
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck) and m.bn3.weight is not None:
                    nn.init.constant_(m.bn3.weight, 0)  # type: ignore[arg-type]
                elif isinstance(m, BasicBlock) and m.bn2.weight is not None:
                    nn.init.constant_(m.bn2.weight, 0)  # type: ignore[arg-type]

    def _make_layer(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        planes: int,
        blocks: int,
        stride: int = 1,
        dilate: bool = False,
    ) -> nn.Sequential:
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation, norm_layer
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return nn.Sequential(*layers)

    def _forward_impl(self, x: Tensor) -> Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

    def forward(self, x: Tensor) -> Tensor:
        return self._forward_impl(x)

class ResNet34_Gradual_Unfreezing(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.backbone = ResNet(BasicBlock, [3, 4, 6, 3])

        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def freeze_backbone(self):
        for name, param in self.backbone.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False

    def unfreeze_layer(self, layer_name: str):
        for name, param in self.backbone.named_parameters():
            if layer_name in name:
                param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


# Training Loop

In [3]:
import torch
import torch.nn as nn
import os
import json
import time
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

        self.run_name = config["run_name"]
        self.save_dir = config["save_dir"]
        os.makedirs(self.save_dir, exist_ok=True)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = self._get_optimizer()
        self.scheduler = self._get_scheduler()

        self.start_epoch = 1
        self.history = self._init_history()

        resume_from = config.get("resume_from")
        resume_epoch = config.get("resume_epoch")

        if (resume_from is None) != (resume_epoch is None):
            raise ValueError("resume_from and resume_epoch must both be provided together, or neither.")

        if resume_from and resume_epoch:
            self._resume(resume_from, resume_epoch)

    def _init_history(self):
        return {
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "lr": [],
            "epoch_duration_seconds": [],
            "unfrozen_layers": [],
            "confusion_matrix": [],
            "classification_report": [],
            "meta": {
                "run_name": self.run_name,
                "resumed_from": None,
                "resumed_from_epoch": None,
                "model_class": self.model.__class__.__name__,
                "num_classes": self.config.get("num_classes", None),
                "config": self.config
            }
        }

    def _save_history(self):
        path = os.path.join(self.save_dir, f"{self.run_name}_history.json")
        with open(path, 'w') as f:
            json.dump(self.history, f, indent=2)

    def _get_optimizer(self):
        optimizer_name = self.config.get("optimizer", "adamw").lower()
        lr = self.config.get("lr", 1e-3)
        weight_decay = self.config.get("weight_decay", 1e-4)
        trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())

        if optimizer_name == "adamw":
            return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
        elif optimizer_name == "sgd":
            return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
        else:
            raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    def _get_scheduler(self):
        scheduler_name = self.config.get("scheduler", "cosine").lower()
        epochs = self.config.get("epochs", 30)

        if scheduler_name == "cosine":
            return CosineAnnealingLR(self.optimizer, T_max=epochs)
        elif scheduler_name == "plateau":
            return ReduceLROnPlateau(self.optimizer, mode='max', patience=3, factor=0.5)
        else:
            raise ValueError(f"Unsupported scheduler: {scheduler_name}")

    def _find_checkpoint(self, run_name, target_epoch):
        available = []
        for fname in os.listdir(self.save_dir):
            prefix = f"{run_name}_epoch_"
            if fname.startswith(prefix) and fname.endswith(".pth"):
                try:
                    epoch_num = int(fname[len(prefix):-4])
                    if epoch_num <= target_epoch:
                        available.append(epoch_num)
                except ValueError:
                    continue

        if not available:
            raise FileNotFoundError(
                f"No checkpoints found for run '{run_name}' at or before epoch {target_epoch} in {self.save_dir}"
            )

        best_epoch = max(available)
        path = os.path.join(self.save_dir, f"{run_name}_epoch_{best_epoch}.pth")
        print(f"Found checkpoint: {path} (requested epoch {target_epoch})")
        return path, best_epoch

    def _save_checkpoint(self, epoch, tag):
        filename = f"{self.run_name}_{tag}.pth"
        path = os.path.join(self.save_dir, filename)
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "history": self.history
        }, path)
        print(f"Checkpoint saved: {path}")

    def _resume(self, resume_from, resume_epoch):
        ckpt_path, actual_epoch = self._find_checkpoint(resume_from, resume_epoch)
        checkpoint = torch.load(ckpt_path, map_location=self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"])

        if self.run_name == resume_from:
            try:
                self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            except ValueError:
                print("Optimizer state mismatch, starting fresh optimizer (expected when freeze state differs)")
        else:
            print("Cross-run resume detected, skipping optimizer state — starting fresh optimizer")

        source_history = checkpoint["history"]
        per_epoch_keys = ["train_loss", "val_loss", "train_acc", "val_acc",
                          "lr", "epoch_duration_seconds", "unfrozen_layers"]

        for key in per_epoch_keys:
            if key in source_history:
                self.history[key] = source_history[key][:actual_epoch]

        for key in ["confusion_matrix", "classification_report"]:
            if key in source_history:
                self.history[key] = [
                    entry for entry in source_history[key]
                    if entry["epoch"] <= actual_epoch
                ]

        self.history["meta"]["resumed_from"] = resume_from
        self.history["meta"]["resumed_from_epoch"] = actual_epoch

        self.start_epoch = actual_epoch + 1
        print(f"Resumed from '{resume_from}' epoch {actual_epoch}, continuing as '{self.run_name}' from epoch {self.start_epoch}")

    def _run_epoch(self, loader, is_training):
        self.model.train() if is_training else self.model.eval()

        total_loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.set_grad_enabled(is_training):
            current_batch = 0
            for images, labels in loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                if is_training:
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item()
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                current_batch += 1

        return total_loss / len(loader), correct / total, all_preds, all_labels

    def train(self, train_loader, val_loader, class_names=None):
        epochs = self.config.get("epochs", 30)
        save_every = self.config.get("save_every", 5)
        eval_every = self.config.get("eval_every", 5)
        unfreeze_at = self.config.get("unfreeze_at_epoch", None)
        unfreeze_layer = self.config.get("unfreeze_layer", "layer4")
        best_val_acc = max(self.history["val_acc"]) if self.history["val_acc"] else 0.0

        for epoch in range(self.start_epoch, epochs + 1):
            epoch_start = time.time()
            print(f"\nEpoch {epoch}/{epochs}")

            if unfreeze_at and epoch == unfreeze_at:
                self.model.unfreeze_layer(unfreeze_layer)
                self.optimizer = self._get_optimizer()
                self.scheduler = self._get_scheduler()
                print(f"Unfroze {unfreeze_layer}, reinitialized optimizer")

            unfrozen = [
                name for name, param in self.model.named_parameters()
                if param.requires_grad
            ]
            self.history["unfrozen_layers"].append(unfrozen)

            train_loss, train_acc, _, _ = self._run_epoch(train_loader, is_training=True)
            val_loss, val_acc, val_preds, val_labels = self._run_epoch(val_loader, is_training=False)

            if isinstance(self.scheduler, ReduceLROnPlateau):
                self.scheduler.step(val_acc)
            else:
                self.scheduler.step()

            current_lr = self.optimizer.param_groups[0]['lr']
            epoch_duration = time.time() - epoch_start

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)
            self.history["lr"].append(current_lr)
            self.history["epoch_duration_seconds"].append(epoch_duration)

            print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
            print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
            print(f"  LR: {current_lr:.6f} | Duration: {epoch_duration:.1f}s")

            if epoch % eval_every == 0:
                cm = confusion_matrix(val_labels, val_preds)
                self.history["confusion_matrix"].append({"epoch": epoch, "matrix": cm.tolist()})

                if class_names:
                    report = classification_report(
                        val_labels, val_preds,
                        target_names=class_names,
                        output_dict=True
                    )
                    self.history["classification_report"].append({"epoch": epoch, "report": report})
                    print(f"  Per-class report saved for epoch {epoch}")

            self._save_history()

            if epoch % save_every == 0:
                self._save_checkpoint(epoch, tag=f"epoch_{epoch}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                self._save_checkpoint(epoch, tag="best")
                print(f"  New best model (val_acc: {val_acc:.4f})")

        final_path = os.path.join(self.save_dir, f"{self.run_name}_final.pth")
        torch.save(self.model.state_dict(), final_path)
        print(f"\nFinal weights saved: {final_path}")
        print(f"Best val accuracy: {best_val_acc:.4f}")

        return self.history


# Run

In [ ]:
import torch

def main():
    config = {
        "run_name": "resnet34_new_augment_v1",
        "save_dir": "/kaggle/working//checkpoints",
        "epochs": 10,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "optimizer": "adamw",
        "scheduler": "cosine",
        "save_every": 5,
        "eval_every": 5,
        "unfreeze_at_epoch": 3,
        "unfreeze_layer": "layer4",
        "num_classes": 9,
        "batch_size": 32
    }

    dataset_path = "/kaggle/input/datasets/sarutsomprasert/augmentedv2/output_2_resized" 

    print(f"Loading data from {dataset_path}...")
    try:
        train_loader, val_loader, class_idx = get_dataloader(dataset_path, config)
        print(f"Data loaded. Classes: {class_idx}")
    except Exception as e:
        print(f"Error loading data: {e}")
        return

    print("Initializing model...")
    model = ResNet34_Gradual_Unfreezing(num_classes=config["num_classes"])
    
    # Weight path - User should update this
    weight_path = "/kaggle/input/datasets/sarutsomprasert/imagenetweight/resnet34-b627a593.pth"
    if torch.os.path.exists(weight_path):
        print(f"Loading weights from {weight_path}...")
        model = load_weights(model=model, weight_path=weight_path)
    else:
        print(f"Weight file {weight_path} not found. Starting with random weights.")

    model.freeze_backbone()
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")

    print("Starting training...")
    trainer = Trainer(model, config)
    history = trainer.train(train_loader, val_loader, class_names=list(class_idx.keys()))
    
    print("Training complete.")

# main()

In [ ]:
import shutil
import os

# Copy v1 checkpoints from input to working so Trainer can find them
src = "/kaggle/input/datasets/sarutsomprasert/firstcheckpoint/checkpoints"
dst = "/kaggle/working/checkpoints"
os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    print(f"Copied: {f}")

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image

class AlbumentationsDataset(torch.utils.data.Dataset):
    def __init__(self, image_folder_dataset, transform):
        self.dataset = image_folder_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        # img is a PIL image from ImageFolder
        img_np = np.array(img)
        augmented = self.transform(image=img_np)
        return augmented["image"], label

normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

real_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Perspective(scale=(0.05, 0.10), p=0.6),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.RandomBrightnessContrast(
        brightness_limit=0.1, 
        contrast_limit=0.1, 
        p=0.3
    ),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

digital_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

def get_mixed_dataloader(real_path, digital_path, config, digital_sample_per_class=100):
    real_raw = ImageFolder(root=real_path)
    digital_raw = ImageFolder(root=digital_path)

    from collections import defaultdict
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(digital_raw.samples):
        class_indices[label].append(idx)

    sampled_indices = []
    for label, indices in class_indices.items():
        sampled = indices[:digital_sample_per_class]
        sampled_indices.extend(sampled)

    digital_subset = torch.utils.data.Subset(digital_raw, sampled_indices)

    real_dataset = AlbumentationsDataset(real_raw, real_transform)
    digital_dataset = AlbumentationsDataset(digital_subset, digital_transform)

    combined = ConcatDataset([real_dataset, digital_dataset])

    real_weight = 5.0
    digital_weight = 1.0
    weights = (
        [real_weight] * len(real_dataset) +
        [digital_weight] * len(digital_dataset)
    )
    sampler = WeightedRandomSampler(weights, num_samples=len(combined), replacement=True)

    os.makedirs(config['save_dir'], exist_ok=True)
    mapping_path = os.path.join(config['save_dir'], f"{config['run_name']}_class_mapping.json")
    with open(mapping_path, 'w') as f:
        json.dump(real_raw.class_to_idx, f, indent=2)
    print(f"Classes: {real_raw.class_to_idx}")
    print(f"Real images: {len(real_dataset)} | Digital sampled: {len(digital_dataset)}")

    train_loader = DataLoader(
        combined,
        batch_size=config['batch_size'],
        sampler=sampler,
        num_workers=2,
        pin_memory=True
    )

    val_transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    val_dataset = AlbumentationsDataset(real_raw, val_transform)
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, val_loader, real_raw.class_to_idx


def main_v2():
    config_v2 = {
        "run_name": "resnet34_v3",
        "resume_from": "resnet34_v1",
        "resume_epoch": 10,              # use whatever your best checkpoint epoch is
        "save_dir": "/kaggle/working/checkpoints",
        "epochs": 20,                    # 20 more epochs on top of v1
        "lr": 1e-4,                      # much lower than v1
        "weight_decay": 1e-4,
        "optimizer": "adamw",
        "scheduler": "cosine",
        "save_every": 5,
        "eval_every": 5,
        "unfreeze_at_epoch": 9999,          # unfreeze layer3 sooner
        "unfreeze_layer": "layer4",
        "num_classes": 9,
        "batch_size": 32
    }

    real_path = "/kaggle/input/datasets/sarutsomprasert/realcard/DOMAIN EXPANSION"      # update this path
    digital_path = "/kaggle/input/datasets/sarutsomprasert/ygoaugmented/output_resized"

    print("Loading mixed dataset...")
    train_loader, val_loader, class_idx = get_mixed_dataloader(
        real_path, digital_path, config_v2, digital_sample_per_class=100
    )

    print("Initializing model...")
    model = ResNet34_Gradual_Unfreezing(num_classes=9)
    model.freeze_backbone()

    print("Starting v2 fine-tuning...")
    trainer = Trainer(model, config_v2)
    history = trainer.train(train_loader, val_loader, class_names=list(class_idx.keys()))

    print("Fine-tuning complete.")

main_v2()